In [7]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import plotly.graph_objects as go  # Add this line
from plotly.subplots import make_subplots
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields,apply_conservative_classification


In [8]:
ss = [
    {'solution_folder': f"RTS-GMLC_v2.4.1", 'net_demand_type': 'net_load'},

    
]
demand= []
random_demand = []
reserve = []
# energy_reserve = []
for s in ss:
    demand_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'uc','Demand.csv'))
    random_demand_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'ed','random_demand.csv'))
    reserve_ = pd.read_csv(os.path.join("..", "input", s['solution_folder'], 'uc','Reserve.csv'))
    demand_  = add_fields(demand_, net_demand_type = s['net_demand_type'])
    random_demand_  = add_fields(random_demand_, net_demand_type = s['net_demand_type'])
    reserve_  = add_fields(reserve_, net_demand_type = s['net_demand_type'])
    # energy_reserve_ = pd.read_csv(os.path.join("..", "input", input_file, 'uc','Energy reserve.csv')) 
    demand.append(demand_)
    random_demand.append(random_demand_)
    reserve.append(reserve_)
    # energy_reserve.append(energy_reserve_)

demand = pd.concat(demand).set_index(['net_demand_type', 'day', 'hour']).sort_index()
random_demand = pd.concat(random_demand).set_index([ 'net_demand_type', 'day', 'hour']).sort_index()
reserve = pd.concat(reserve).set_index(['net_demand_type', 'day', 'hour']).sort_index()

# random_demand = filter_demand(demand, random_demand, reserve)

imbalance = random_demand.sub(demand['demand'], axis=0, level=['net_demand_type', 'day','hour'])


TypeError: Join on level between two MultiIndex objects is ambiguous

In [ ]:
read_files = False
write_files = False
solution_keys = ['demand','reserve', 'energy_reserve']

if not read_files:
    ss = [
        # {'solution_folder': f"RTS-GMLC_v18.3s", 'model_type' : 'envelope'},
        # {'solution_folder': f"RTS-GMLC_v19.4s", 'model_type' : 'e-reserve'},
        {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
        {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]
    days = range(1,2)
    s_uc = []
    s_ed = []
    gcd_KPI_adequacy = []
    gcdi_KPI_adequacy = []
    

    for sol in ss:
        # ρ = sol['ρ']
        s = sol['solution_folder']
        # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
        # s_ed_name = 's_sed'
        s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
        if sol['model_type'] != 'stochastic':
            s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
        else:
            s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
        s_uc.append(s_uc_)
        s_ed.append(s_ed_)

        # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
        # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

        # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
        # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

        # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
        # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

    s_uc = combine_solutions(s_uc)
    s_ed = combine_solutions(s_ed)
    # gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
    # gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

    for k,v in s_uc.items():
        if 'µ' in v.columns:
            s_uc[k]['model_type'] =  v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    for k,v in s_ed.items():
        if 'µ' in v.columns:
            s_ed[k]['model_type'] = v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    # if 'µ' in gcdi_KPI_adequacy.columns: 
    #     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    #     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    if write_files:
        for k,v in s_uc.items():
            v.to_csv(f's_uc_{k}.csv', index=False)
        for k,v in s_ed.items():
            v.to_csv(f's_ed_{k}.csv', index=False)

else:
    s_uc = {}
    s_ed = {}
    for solution_key in solution_keys:
        s_uc[solution_key] = pd.read_csv(f's_uc_{solution_key}.csv')
        # s_ed[solution_key] = pd.read_csv(f's_ed_{solution_key}.csv')    

    # gcd_KPI_adequacy = pd.read_csv('gcd_KPI_adequacy.csv', index_col=0)
    # gcdi_KPI_adequacy = pd.read_csv('gcdi_KPI_adequacy.csv', index_col=0)







In [ ]:
s_uc['reserve']['model_type'].unique()

array(['envelope'], dtype=object)

In [ ]:
s_ed['demand']

,hour,demand_MW,r_id,resource,LOL_MW,LGEN_MW,iteration,day,configuration,µ,model_type,solution_id
0,1,800.961098,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
1,2,751.592509,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
2,3,811.027625,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
3,4,811.472114,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
4,5,870.674819,None,system,0.0,22.604868,demand_1,1,base_ramp_storage_envelopes_mu_1,mu_1,envelope,RTS-GMLC_v32.3s
...,...,...,...,...,...,...,...,...,...,...,...,...
19,20,3353.436483,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s
20,21,3098.238081,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s
21,22,2792.303357,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s
22,23,2678.503522,None,system,0.0,0.000000,demand_1,1,base_ramp_storage_envelopes_up_1_dn_1,1.0,e-reserve,RTS-GMLC_v32.1s


In [ ]:
imbalance

demand_1    demand_2    demand_3    demand_4  \
net_demand_type day hour                                                   
net_load        1   1     430.993435 -209.360826  -81.479679 -231.285999   
                    2     558.067093  148.711596  384.036536 -335.300875   
                    3     279.145699 -136.969532  344.344526 -490.166545   
                    4     548.722641 -266.117375   86.902790 -600.989112   
                    5     293.859878 -502.494104 -253.237851 -701.590613   
...                              ...         ...         ...         ...   
                365 8756 -248.050083  657.309551 -594.130852  121.225583   
                    8757 -116.141998  369.907198 -382.486861 -150.679384   
                    8758  271.318493   21.945057 -195.421781  177.842095   
                    8759  321.574654 -104.641407 -145.301935  215.232657   
                    8760  375.796655 -192.528018 -453.536646  -85.407122   

                            demand_5    demand_6     demand_7    demand_8  \
net_demand_type day hour                                                    
net_load        1   1     235.242881  280.912500   274.764506 -183.654191   
                    2      49.796379 -227.554814  -131.046756  -87.581827   
                    3      99.865279   38.295356   -82.892082 -267.117376   
                    4     206.530850 -242.449069  -453.474171 -215.458350   
                    5     305.366228  223.747209  -285.387490 -164.299986   
...                              ...         ...          ...         ...   
                365 8756  984.894641  525.957680  1294.715954 -230.847414   
                    8757  737.031382  530.827323  1375.601843 -336.528423   
                    8758  460.599461 -154.516993  1027.926246 -324.476342   
                    8759  501.458259  -97.982287   628.182074 -476.607558   
                    8760  589.306392  350.677448   875.055898 -223.234972   

                            demand_9   demand_10  ...  demand_491  demand_492  \
net_demand_type day hour                          ...                           
net_load        1   1     290.603118  409.665906  ...  650.920828 -396.776350   
                    2     482.695653  397.010516  ...  562.326903 -439.078092   
                    3     269.164577  311.092899  ...  579.149906 -682.136037   
                    4     328.542720  320.632577  ...  505.747325 -580.054594   
                    5     299.209815  331.743898  ...  420.077606 -724.347539   
...                              ...         ...  ...         ...         ...   
                365 8756  420.888513  -45.573397  ... -186.476905  281.947954   
                    8757  923.640717 -328.511541  ...   55.343172 -166.154746   
                    8758  309.159350   19.678882  ...  274.745548  299.403825   
                    8759  275.866711 -263.291536  ...   -0.614695  -11.146602   
                    8760    4.836826 -339.912584  ...   55.185632  -20.490077   

                          demand_493  demand_494  demand_495  demand_496  \
net_demand_type day hour                                                   
net_load        1   1    -211.071822  431.565653  678.180688  151.164422   
                    2     132.738971  423.667978  407.427014  137.001070   
                    3     200.598896  455.859806  291.983774  819.643015   
                    4     162.679172 -129.451693   94.229482  907.061121   
                    5     191.490748  338.162357 -282.526279  757.465960   
...                              ...         ...         ...         ...   
                365 8756 -254.696786  387.572852  713.217935 -252.398252   
                    8757 -208.258380  186.171793  546.276069   22.859405   
                    8758 -235.696652 -356.051674  546.285317 -135.215623   
                    8759 -598.423783 -144.201478  433.663360 -189.682963   
                    8760 -448.008660 -386.943353  452.774590 -176.11726